<a href="https://colab.research.google.com/github/jayanthkumarmadduri-code/Data-Structures-Practice/blob/main/RouteTour.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Route Tour Planner**


# **YBI Foundation**

**Submitted by:
Jayanth Kumar**

## Objective

The objective of this project is to analyze travel route data and develop an intelligent travel planning system that helps users understand route patterns, traffic conditions, and travel efficiency through data visualization and analysis. The project aims to explore different aspects of travel data using Python libraries and interactive visualizations, enabling better route planning and informed decision-making for travelers. This project also demonstrates the application of AI-based data analysis techniques in solving real-world transportation and tourism challenges.

In [1]:
# Import the necessary files
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

In [ ]:
from google.colab import files
import pandas as pd

# Upload the file if it's not already in the Colab environment
print("Please upload the 'SmartTourRoutePlanner.csv' file:")
uploaded = files.upload()

# Check if the file was uploaded
if 'SmartTourRoutePlanner.csv' in uploaded:
  print('File "SmartTourRoutePlanner.csv" uploaded successfully.')
  df = pd.read_csv('SmartTourRoutePlanner.csv')
  print('DataFrame loaded successfully.')
else:
  print('Error: "SmartTourRoutePlanner.csv" was not uploaded. Please ensure you upload the correct file.')

Please upload the 'SmartTourRoutePlanner.csv' file:


# **Loading The Dataset**

In [ ]:
df

In [ ]:
df.shape

In [ ]:
df.size

In [ ]:
df.columns

In [ ]:
df.info

In [ ]:
df.describe

In [ ]:
df.info

In [ ]:
df.duplicated().sum()

In [ ]:
df.isnull().sum()

In [ ]:
df['start_location'].value_counts(normalize=True)

In [ ]:
df['end_location'].value_counts(normalize=True)

# **EXPLORATORY DATA ANALYSIS**

In [ ]:
import pandas as pd
import folium

city_coords = {
    "Delhi": (28.6139, 77.2090),
    "Mumbai": (19.0760, 72.8777),
    "Bangalore": (12.9716, 77.5946),
    "Chennai": (13.0827, 80.2707),
    "Kolkata": (22.5726, 88.3639),
    "Agra": (27.1767, 78.0081),
    "Goa": (15.2993, 74.1240),
    "Shimla": (31.1048, 77.1734),
    "Ooty": (11.4064, 76.6932),
    "Mahabalipuram": (12.6208, 80.1937)
}

df["start_lat"]=df["start_location"].map(lambda x: city_coords.get(x, (None, None))[0])
df["start_lon"]=df["start_location"].map(lambda x: city_coords.get(x, (None, None))[1])

df["end_lat"]=df["end_location"].map(lambda x: city_coords.get(x, (None, None))[0])
df["end_lon"]=df["end_location"].map(lambda x: city_coords.get(x, (None, None))[1])

m=folium.Map(location=[22.5, 80], zoom_start=5, tiles="cartodbpositron")

mode_colors={
    "car":"blue",
    "train":"green",
    "bike":"orange",
    "walk":"purple",
    "bus":"brown"
}

for _, row in df.iterrows():

    if pd.notnull(row["start_lat"]) and pd.notnull(row["end_lat"]):

        start=(row["start_lat"], row["start_lon"])
        end=(row["end_lat"], row["end_lon"])

        color=mode_colors.get(str(row["transport_mode"]).lower(), "black")

        folium.PolyLine(
            locations=[start, end],
            color=color,
            weight=3,
            opacity=0.8,
            tooltip=f"{row['start_location']} → {row['end_location']} ({row['transport_mode']})"
        ).add_to(m)

        folium.CircleMarker(
            location=start,
            radius=row["popularity_score"] / 20,
            color="blue",
            fill=True,
            fill_opacity=0.7,
            popup=f"Start: {row['start_location']}"
        ).add_to(m)

        folium.CircleMarker(
            location=end,
            radius=row["popularity_score"] / 20,
            color="orange",
            fill=True,
            fill_opacity=0.7,
            popup=f"End: {row['end_location']}"
        ).add_to(m)

m.save("travel_route_map.html")
m

**Interactive Travel Route Map (Geographic Route Visualization) Routes are represented as connecting lines between origin and destination cities, while markers indicate the city locations. The size of each marker reflects the popularity of the travel routes associated with that city. By examining this map, we can understand the geographic connectivity between cities, identify popular travel routes, and observe how tourism or travel demand is distributed across different regions of India.**

In [ ]:
import pandas as pd
import plotly.graph_objects as go

entry_fee = df['entry_fee'].mean()
accommodation = df['accommodation_cost'].mean()
food = df['food_cost'].mean()

labels = [
    "Total Travel Cost",
    f"Entry Fee ({entry_fee:.2f})",
    f"Accommodation ({accommodation:.2f})",
    f"Food ({food:.2f})"
]

fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=25,
        thickness=30,
        line=dict(color="black", width=0.5),
        label=labels,
        color=["#4C78A8","#F58518","#54A24B","#B279A2"]
    ),

    link=dict(
        source=[0,0,0],
        target=[1,2,3],
        value=[entry_fee, accommodation, food],
        color=[
            "rgba(245,133,24,0.5)",
            "rgba(84,162,75,0.5)",
            "rgba(178,121,162,0.5)"
        ]
    )
)])

fig.update_layout(
    title="Travel Cost Flow Distribution",
    font_size=13,
    width=900,
    height=500
)

fig.show()

**Sankey Diagram – Travel Cost Flow Distribution This visualization illustrates how the total travel cost is distributed across different expense categories. The flows represent the contributions of entry fees, accommodation costs, and food expenses to the overall travel cost. By analyzing this diagram, it becomes easier to understand which expense category contributes the most to the total trip cost and how travel expenses are proportionally allocated.**

In [ ]:
import pandas as pd
import plotly.express as px

df['traffic_density_bin']=pd.cut(
    df['traffic_density'],
    bins=[0, 0.2, 0.4, 0.6, 0.8, 1],
    labels=['Very Low', 'Low', 'Medium', 'High', 'Very High']
)

traffic_time=df.groupby('traffic_density_bin')['estimated_travel_time_hr'].mean().reset_index()

fig=px.line(
    traffic_time,
    x='traffic_density_bin',
    y='estimated_travel_time_hr',
    markers=True,
    title='Impact of Traffic Density on Estimated Travel Time',
    labels={
        'traffic_density_bin': 'Traffic Density Level',
        'estimated_travel_time_hr': 'Average Travel Time (hours)'
    }
)

fig.update_layout(
    title_x=0.5,
    xaxis_title="Traffic Density Level",
    yaxis_title="Average Travel Time (Hours)"
)

fig.show()

**Line Plot – Impact of Traffic Density on Travel Time
This line plot illustrates the relationship between traffic density levels and the estimated travel time. Traffic density values are grouped into categories ranging from Very Low to Very High, and the average travel time is calculated for each level. The plot helps visualize how increasing traffic congestion influences travel duration. By observing this trend, we can understand how higher traffic density generally leads to longer travel times, highlighting the importance of traffic conditions in travel route planning and time estimation.**

In [ ]:
import plotly.express as px


fig = px.density_heatmap(
    df,
    x="season",
    y="day_type",
    z="popularity_score",
    histfunc="avg",
    animation_frame="transport_mode",
    text_auto=True,
    title="Heatmap of Travel Demand (Day Type vs Season)",

    color_continuous_scale=[
        [0, "#f7fbff"],
        [0.2, "#c6dbef"],
        [0.4, "#6baed6"],
        [0.6, "#3182bd"],
        [0.8, "#08519c"],
        [1, "#08306b"]
    ]
)
fig["layout"].pop("updatemenus")

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    xaxis_title="Season",
    yaxis_title="Day Type",
    coloraxis_colorbar=dict(title="Travel Demand")
)

fig.update_traces(
    hovertemplate="Season: %{x}<br>Day Type: %{y}<br>Demand: %{z}<extra></extra>"
)

fig.show()

**Heatmap – Travel Demand by Season and Day Type
This heatmap visualizes the average travel demand across different seasons and types of days. Color intensity represents the level of travel popularity, where darker shades indicate higher demand. By examining this visualization, we can identify periods of peak travel activity and better understand seasonal tourism patterns as well as differences between weekday and weekend travel behavior.**

In [ ]:
import pandas as pd
import plotly.express as px
sunburst_data = df.groupby(
    ["season", "transport_mode", "destination_type"]
)["popularity_score"].sum().reset_index()

season_order = ["Winter", "Spring", "Summer", "Monsoon", "Autumn"]

sunburst_data["season"] = pd.Categorical(
    sunburst_data["season"],
    categories=season_order,
    ordered=True
)

sunburst_data = sunburst_data.sort_values("season")

fig = px.sunburst(
    sunburst_data,
    path=["season", "transport_mode", "destination_type"],
    values="popularity_score",
    color="season",
    color_discrete_sequence=px.colors.qualitative.Set2,
    title="Travel Preference Hierarchy: Season → Transport Mode → Destination Type"
)

fig.update_layout(
    title_x=0.5,
    margin=dict(t=50, l=25, r=25, b=25),
    template="plotly_white"
)

fig.update_traces(
    hovertemplate="<b>%{label}</b><br>Popularity Score: %{value}<extra></extra>"
)

fig.show()

**Sunburst Chart – Travel Preference Hierarchy
This sunburst chart visualizes the hierarchical structure of travel preferences based on season, transport mode, and destination type. The inner layer represents the different seasons, followed by the transport modes used during those seasons, and finally the types of destinations chosen by travelers. The size of each segment reflects the aggregated popularity score, indicating how common each travel combination is. By analyzing this visualization, we can observe how travel choices vary across seasons, identify preferred transport modes, and understand the types of destinations that attract travelers under different seasonal conditions.**

In [ ]:
import pandas as pd
import plotly.express as px

fig = px.scatter(
    df,
    x="user_budget",
    y="satisfaction_rating",
    color="transport_mode",
    symbol="season",
    title="User Budget vs Satisfaction Rating by Transport Mode",

    labels={
        "user_budget": "User Budget",
        "satisfaction_rating": "Satisfaction Rating",
        "transport_mode": "Transport Mode",
        "season": "Season"
    },

    color_discrete_map={
"Car": "#1f77b4",
"Train": "#2ca02c",
"Bike": "#ff7f0e",
"Walk": "#9467bd",
"Bus": "#d62728"
},

    hover_data=[
        "start_location",
        "end_location",
        "season",
        "transport_mode",
        "satisfaction_rating"
    ]
)

fig.update_layout(
    title_x=0.5,
    xaxis_title="User Budget",
    yaxis_title="Satisfaction Rating"
)

fig.show()

**Scatter Plot – User Budget vs Satisfaction Rating
This scatter plot illustrates the relationship between the user’s travel budget and their satisfaction rating for different travel routes. Each point represents an individual travel instance, where the position indicates the budget allocated for the trip and the corresponding satisfaction level reported by the traveler. The points are further differentiated by transport mode and seasonal variations, allowing for clearer comparisons across travel conditions. By analyzing this visualization, we can explore whether higher travel budgets tend to lead to higher satisfaction levels and observe how different transport options influence overall travel experience.**

## Conclusion

This project successfully analyzed travel route data using Python and various data visualization techniques. Through data preprocessing, exploratory data analysis, and interactive visualizations such as maps, heatmaps, Sankey diagrams, and sunburst charts, valuable insights into travel patterns and route efficiency were obtained. The analysis helps identify important trends that can support smarter travel planning and improved route selection. Overall, the project demonstrates how AI-driven data analysis can assist in understanding transportation data and lays the foundation for future enhancements using Generative AI, such as personalized travel recommendations and AI-generated trip planning.

## Future Scope

In future, this project can be enhanced by integrating Generative AI models such as Google's Gemini or OpenAI's GPT to generate personalized travel itineraries, recommend tourist destinations based on user preferences, answer travel-related queries through a chatbot, and provide intelligent route suggestions in natural language. These enhancements would make SmartRouteTour a complete AI-powered travel assistant.